# i3X API Test Notebook
**Industrial Information Interoperability eXchange (i3X) — CESMII**

This notebook tests every endpoint documented at `https://api.i3x.dev/v0/docs`.  
You can retarget it to any i3X-compliant server by changing `BASE_URL` in **Cell 1**.

> **Demo endpoint** (public, no auth required): `https://api.i3x.dev/v0`

---
## Endpoint Coverage

| # | Group | Endpoint | Method |
|---|-------|----------|--------|
| 1 | Discovery | `/namespaces` | GET |
| 2 | Discovery | `/objecttypes` | GET |
| 3 | Objects | `/objects` | GET |
| 4 | Objects | `/objects/{id}` | GET |
| 5 | Objects | `/objects/list` | POST |
| 6 | Objects | `/objects/value` | POST |
| 7 | Objects | `/objects/history` | POST |
| 8 | Objects | `/objects/related` | POST |
| 9 | Subscriptions | `POST /subscriptions` | POST |
| 10 | Subscriptions | `POST /subscriptions/{id}/register` | POST |
| 11 | Subscriptions | `POST /subscriptions/{id}/sync` | POST |
| 12 | Subscriptions | `GET /subscriptions/{id}/stream` | GET (SSE) |
| 13 | Health | `/health` (best-effort) | GET |


## Cell 1 — Configuration
Change `BASE_URL` to point at any i3X-compliant server. Leave `BEARER_TOKEN` empty for the public demo.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Change BASE_URL to target a different i3X server
BASE_URL = "https://api.i3x.dev/v0"

# Bearer token — leave empty string for the public demo (no auth required)
BEARER_TOKEN = ""

# Request timeout in seconds
TIMEOUT = 15

# ── DERIVED ───────────────────────────────────────────────────────────────────
BASE_URL = BASE_URL.rstrip("/")

AUTH_HEADERS = {}
if BEARER_TOKEN:
    AUTH_HEADERS["Authorization"] = f"Bearer {BEARER_TOKEN}"

JSON_HEADERS = {**AUTH_HEADERS, "Content-Type": "application/json", "Accept": "application/json"}
GET_HEADERS  = {**AUTH_HEADERS, "Accept": "application/json"}

print(f"✅  Target server : {BASE_URL}")
print(f"✅  Auth header   : {'Bearer ***' if BEARER_TOKEN else '(none — public demo)'}")


## Cell 2 — Helper Utilities
Pretty-print responses, collect results across runs.

In [ ]:
import requests, json, pprint, textwrap
from datetime import datetime, timezone, timedelta
from IPython.display import display, JSON, Markdown

# Accumulated results so IDs discovered early can be reused by later cells
_state = {
    "namespace_uris": [],
    "object_type_ids": [],
    "object_ids": [],
    "subscription_id": None,
}

def _show(label, resp):
    """Display a labelled, pretty-printed response."""
    status_emoji = "✅" if resp.ok else "❌"
    print(f"\n{status_emoji}  {label}")
    print(f"    HTTP {resp.status_code}  |  {resp.elapsed.total_seconds():.3f}s  |  {len(resp.content)} bytes")
    try:
        body = resp.json()
        display(JSON(body))
    except Exception:
        print(textwrap.shorten(resp.text, 400))
    return resp

def _safe_get(url, **kwargs):
    try:
        return requests.get(url, headers=GET_HEADERS, timeout=TIMEOUT, **kwargs)
    except requests.RequestException as e:
        print(f"❌  Request failed: {e}")
        return None

def _safe_post(url, payload, **kwargs):
    try:
        return requests.post(url, headers=JSON_HEADERS, json=payload, timeout=TIMEOUT, **kwargs)
    except requests.RequestException as e:
        print(f"❌  Request failed: {e}")
        return None

print("✅  Helpers loaded")


## Test 1 — Health Check
`GET /health`  
Not formally documented in the spec but conventional for REST APIs.

In [ ]:
# ── GET /health ───────────────────────────────────────────────────────────────
resp = _safe_get(f"{BASE_URL}/health")
if resp:
    _show("GET /health", resp)
else:
    print("ℹ️  /health endpoint not present — this is optional per the spec.")


## Test 2 — List Namespaces
`GET /namespaces`  
Discovers all namespace URIs available on this server. Results are saved for later use.

In [ ]:
# ── GET /namespaces ───────────────────────────────────────────────────────────
resp = _safe_get(f"{BASE_URL}/namespaces")
if resp:
    _show("GET /namespaces", resp)
    if resp.ok:
        data = resp.json()
        # Spec may return a list directly or {namespaces: [...]}
        ns_list = data if isinstance(data, list) else data.get("namespaces", data.get("value", []))
        for ns in ns_list:
            uri = ns.get("uri") or ns.get("namespaceUri") or (ns if isinstance(ns, str) else None)
            if uri:
                _state["namespace_uris"].append(uri)
        print(f"\n📦  Captured {len(_state['namespace_uris'])} namespace URI(s):")
        for u in _state["namespace_uris"]:
            print(f"    • {u}")


## Test 3 — List Object Types
`GET /objecttypes`  
Optionally filter by namespace URI. Captures type IDs for downstream tests.

In [ ]:
# ── GET /objecttypes ─────────────────────────────────────────────────────────
# Unfiltered
resp_all = _safe_get(f"{BASE_URL}/objecttypes")
if resp_all:
    _show("GET /objecttypes (all)", resp_all)

# Filtered by first namespace if available
if _state["namespace_uris"]:
    ns = _state["namespace_uris"][0]
    resp_ns = _safe_get(f"{BASE_URL}/objecttypes",
                        params={"namespaceUri": ns})
    if resp_ns:
        _show(f"GET /objecttypes?namespaceUri={ns!r}", resp_ns)
        if resp_ns.ok:
            data = resp_ns.json()
            types = data if isinstance(data, list) else data.get("objectTypes", data.get("value", []))
            for t in types[:5]:
                tid = t.get("id") or t.get("objectTypeId")
                if tid:
                    _state["object_type_ids"].append(tid)
            print(f"\n📦  Captured {len(_state['object_type_ids'])} object type ID(s) (first 5)")


## Test 4 — List Objects
`GET /objects`  
Returns root-level manufacturing objects. First few IDs are stored for subsequent tests.

In [ ]:
# ── GET /objects ──────────────────────────────────────────────────────────────
resp = _safe_get(f"{BASE_URL}/objects", params={"includeMetadata": "true"})
if resp:
    _show("GET /objects?includeMetadata=true", resp)
    if resp.ok:
        objs = resp.json()
        _state["object_ids"] = [o["elementId"] for o in objs]
        # Cache full object index for the explorer cell
        _state["object_index"] = {o["elementId"]: o for o in objs}
        print(f"\n📦  {len(_state['object_ids'])} objects captured")
        print(f"    IDs: {_state['object_ids'][:5]}{'...' if len(_state['object_ids']) > 5 else ''}")


## Test 5 — Get Object by ID
`GET /objects/{id}`  
Fetches a single object with its properties and metadata.

In [ ]:
# ── GET /objects/{id} ────────────────────────────────────────────────────────
if not _state["object_ids"]:
    print("⚠️  No object IDs captured yet — run Cell 6 (Test 4) first.")
else:
    oid = _state["object_ids"][0]
    resp = _safe_get(f"{BASE_URL}/objects/{oid}")
    if resp:
        _show(f"GET /objects/{oid}", resp)


## Test 6 — Batch Object Lookup
`POST /objects/list`  
Fetch multiple objects by element IDs in a single request.

In [ ]:
# ── POST /objects/list ────────────────────────────────────────────────────────
if not _state["object_ids"]:
    print("⚠️  No object IDs captured — run Cell 6 first.")
else:
    batch_ids = _state["object_ids"][:3]
    payload = {"elementIds": batch_ids}
    print(f"📤  Payload: {json.dumps(payload, indent=2)}")
    resp = _safe_post(f"{BASE_URL}/objects/list", payload)
    if resp:
        _show("POST /objects/list", resp)


## Test 7 — Read Current Values
`POST /objects/value`  
Returns the latest live/current values for the requested element IDs.  
`maxDepth=0` means infinite child recursion; `1` means no child recursion.

In [ ]:
# ── POST /objects/value ───────────────────────────────────────────────────────
if not _state["object_ids"]:
    print("⚠️  No object IDs — run Cell 6 first.")
else:
    # Shallow read of first object
    payload = {
        "elementIds": [_state["object_ids"][0]],
        "maxDepth": 1
    }
    print(f"📤  Payload: {json.dumps(payload, indent=2)}")
    resp = _safe_post(f"{BASE_URL}/objects/value", payload)
    if resp:
        _show("POST /objects/value (shallow, maxDepth=1)", resp)

    # Deep read of first object
    payload_deep = {
        "elementIds": [_state["object_ids"][0]],
        "maxDepth": 0   # 0 = infinite
    }
    print(f"\n📤  Payload (deep): {json.dumps(payload_deep, indent=2)}")
    resp_deep = _safe_post(f"{BASE_URL}/objects/value", payload_deep)
    if resp_deep:
        _show("POST /objects/value (deep, maxDepth=0)", resp_deep)


## Test 8 — Historical Data
`POST /objects/history`  
Retrieves time-series historian data. Default window: last 1 hour.  
Adjust `START_TIME` / `END_TIME` as needed.

In [ ]:
# ── POST /objects/history ────────────────────────────────────────────────────
if not _state["object_ids"]:
    print("⚠️  No object IDs — run Cell 6 first.")
else:
    now        = datetime.now(timezone.utc)
    START_TIME = (now - timedelta(hours=1)).isoformat()
    END_TIME   = now.isoformat()

    payload = {
        "elementIds": [_state["object_ids"][0]],
        "startTime":  START_TIME,
        "endTime":    END_TIME,
        "maxDepth":   1
    }
    print(f"📤  Window: {START_TIME}  →  {END_TIME}")
    print(f"📤  Payload: {json.dumps(payload, indent=2)}")
    resp = _safe_post(f"{BASE_URL}/objects/history", payload)
    if resp:
        _show("POST /objects/history", resp)


## Test 9 — Related Objects
`POST /objects/related`  
Traverses the graph to find objects related to the target, optionally filtered by relationship type.

In [ ]:
# ── POST /objects/related ─────────────────────────────────────────────────────
if not _state["object_ids"]:
    print("⚠️  No object IDs — run Cell 6 first.")
else:
    # No relationship filter (all relationships)
    payload_all = {
        "elementIds":      [_state["object_ids"][0]],
        "includeMetadata": True
    }
    print(f"📤  Payload (all relationships): {json.dumps(payload_all, indent=2)}")
    resp_all = _safe_post(f"{BASE_URL}/objects/related", payload_all)
    if resp_all:
        _show("POST /objects/related (all)", resp_all)

    # Common relationship types to probe
    for rel_type in ["composition", "isPartOf", "references"]:
        payload_rel = {
            "elementIds":       [_state["object_ids"][0]],
            "relationshiptype": rel_type,
            "includeMetadata":  True
        }
        resp_rel = _safe_post(f"{BASE_URL}/objects/related", payload_rel)
        if resp_rel and resp_rel.ok:
            _show(f"POST /objects/related (type='{rel_type}')", resp_rel)
        # Silently skip 4xx for missing relationship types


## Tests 10–12 — Subscriptions (Create → Register → Sync)
`POST /subscriptions` → `POST /subscriptions/{id}/register` → `POST /subscriptions/{id}/sync`  
The three-step subscription flow for polling-based real-time updates.

In [ ]:
# ── Step 1: POST /subscriptions ──────────────────────────────────────────────
print("── Step 1: Create subscription ──────────────────────────────────────")
resp_create = _safe_post(f"{BASE_URL}/subscriptions", {})
if not resp_create:
    print("❌  Could not create subscription — skipping remaining subscription tests.")
else:
    _show("POST /subscriptions", resp_create)
    if resp_create.ok:
        sub_data = resp_create.json()
        sub_id   = (sub_data.get("subscriptionId")
                    or sub_data.get("id")
                    or sub_data.get("subscription_id"))
        _state["subscription_id"] = sub_id
        print(f"\n📦  subscription_id = {sub_id}")
    else:
        print("⚠️  Subscription creation returned non-OK — check auth or server state.")


In [ ]:
# ── Step 2: POST /subscriptions/{id}/register ────────────────────────────────
print("── Step 2: Register objects on subscription ─────────────────────────")
sub_id = _state["subscription_id"]
if not sub_id:
    print("⚠️  No subscription ID — run the Create cell first.")
elif not _state["object_ids"]:
    print("⚠️  No object IDs — run Cell 6 (Test 4) first.")
else:
    payload = {
        "elementIds": _state["object_ids"][:3],
        "maxDepth":   1
    }
    print(f"📤  Registering {len(payload['elementIds'])} element(s) on subscription {sub_id!r}")
    print(f"📤  Payload: {json.dumps(payload, indent=2)}")
    resp_reg = _safe_post(f"{BASE_URL}/subscriptions/{sub_id}/register", payload)
    if resp_reg:
        _show(f"POST /subscriptions/{sub_id}/register", resp_reg)


In [ ]:
# ── Step 3: POST /subscriptions/{id}/sync ────────────────────────────────────
# Returns and clears queued change notifications (poll-mode alternative to SSE)
print("── Step 3: Sync (poll for queued updates) ───────────────────────────")
sub_id = _state["subscription_id"]
if not sub_id:
    print("⚠️  No subscription ID — run the Create cell first.")
else:
    resp_sync = _safe_post(f"{BASE_URL}/subscriptions/{sub_id}/sync", {})
    if resp_sync:
        _show(f"POST /subscriptions/{sub_id}/sync", resp_sync)


## Test 13 — SSE Stream (Real-time)
`GET /subscriptions/{id}/stream`  
Opens a Server-Sent Events stream. The cell collects events for `SSE_DURATION_SEC` seconds, then closes.

In [ ]:
import requests, json, time, textwrap

SSE_DURATION_SEC = 8

sub_id = _state["subscription_id"]
stream_url = f"{BASE_URL}/subscriptions/{sub_id}/stream"

print(f"🔴  SSE stream: {stream_url}  ({SSE_DURATION_SEC}s window)\n")

events_received = []
deadline = time.time() + SSE_DURATION_SEC

with requests.get(
    stream_url,
    headers={**GET_HEADERS, "Accept": "text/event-stream"},
    stream=True,
    timeout=(5, SSE_DURATION_SEC + 5)
) as resp:
    resp.raise_for_status()
    event_type = "message"
    data_lines = []

    for raw in resp.iter_lines(decode_unicode=True):
        if time.time() > deadline:
            print("⏱  Time limit reached.")
            break

        if raw.startswith("event:"):
            event_type = raw[6:].strip()
        elif raw.startswith("data:"):
            data_lines.append(raw[5:].strip())
        elif raw == "":          # blank line = end of event
            if data_lines:
                data = "\n".join(data_lines)
                events_received.append({"event": event_type, "data": data})
                print(f"  📡  [{event_type}]  {textwrap.shorten(data, 120)}")
                if event_type == "done":
                    break
            event_type = "message"
            data_lines = []

print(f"\n✅  {len(events_received)} event(s) received.")
if events_received:
    display(JSON(events_received))

## Test 14 — Summary Report
Displays a summary of discovered entities and re-runs all GET endpoints against captured IDs for a final sanity check.

In [ ]:
print("=" * 60)
print("  i3X API Test Summary")
print("=" * 60)
print(f"  Server        : {BASE_URL}")
print(f"  Auth          : {'Bearer token' if BEARER_TOKEN else 'None (public demo)'}")
print()
print(f"  Namespaces    : {len(_state['namespace_uris'])}")
for u in _state["namespace_uris"]:
    print(f"    • {u}")
print()
print(f"  Object Types  : {len(_state['object_type_ids'])}")
print(f"  Objects found : {len(_state['object_ids'])}")
if _state["object_ids"]:
    print(f"    First IDs   : {_state['object_ids'][:5]}")
print()
print(f"  Subscription  : {_state['subscription_id'] or 'not created'}")
print("=" * 60)


## Bonus — Interactive Object Explorer
Walk the object tree up to `MAX_DEPTH` levels using the `/objects` and `/objects/{id}` endpoints. Useful for understanding the namespace structure.

In [ ]:
# ── Object Explorer — rewritten for actual i3X schema ────────────────────────
MAX_DEPTH = 2

# Build a lookup from the flat /objects list instead of fetching each node
resp = _safe_get(f"{BASE_URL}/objects", params={"includeMetadata": "true"})
all_objects = {o["elementId"]: o for o in resp.json()}
print(f"📦  {len(all_objects)} objects loaded\n")

def _walk(eid, depth=0, visited=None):
    if visited is None:
        visited = set()
    if eid in visited or depth > MAX_DEPTH:
        return
    visited.add(eid)

    obj = all_objects.get(eid)
    if not obj:
        print("  " * depth + f"⚠️  {eid} (not in index)")
        return

    indent   = "  " * depth
    name     = obj.get("displayName", eid)
    type_id  = obj.get("typeId", "")
    ns       = obj.get("namespaceUri", "")
    print(f"{indent}📦  {name}  [{type_id}]  ns={ns}")

    children = obj.get("relationships", {}).get("HasChildren", [])
    for child_id in children:
        _walk(child_id, depth + 1, visited)

# Start from root objects (parentId == "/")
roots = [o["elementId"] for o in all_objects.values() if o.get("parentId") == "/"]
print(f"Roots: {roots}\n")
for r in roots:
    _walk(r)

## Bonus — Custom / Ad-hoc Request
Send any i3X request with full control over method, path, and body.

In [ ]:
# ── CUSTOM REQUEST ────────────────────────────────────────────────────────────
CUSTOM_METHOD  = "GET"           # GET | POST | PUT | DELETE | PATCH
CUSTOM_PATH    = "/objects"      # path relative to BASE_URL
CUSTOM_PAYLOAD = {}              # for POST/PUT

url = f"{BASE_URL}/{CUSTOM_PATH.lstrip('/')}"
print(f"🚀  {CUSTOM_METHOD} {url}")

if CUSTOM_METHOD.upper() == "GET":
    resp = _safe_get(url)
else:
    resp = _safe_post(url, CUSTOM_PAYLOAD)

if resp:
    _show(f"Custom: {CUSTOM_METHOD} {CUSTOM_PATH}", resp)
